# Dense Matrix Surface Demo

This notebook exercises the finished dense `arb_mat` / `acb_mat` surface in pure JAX:
- four modes (`point`, `basic`, `adaptive`, `rigorous`)
- matrix-RHS solve
- LU-reuse solve plans
- cached matvec plans
- transpose / conjugate-transpose / diag helpers


In [ ]:
import jax
import jax.numpy as jnp

from arbplusjax import acb_core, acb_mat, api, arb_mat, double_interval as di

print('backend:', jax.default_backend())
print('devices:', jax.devices())

In [ ]:
a_mid = jnp.array([[4.0, 1.0, 0.0], [2.0, 3.0, 1.0], [0.0, 1.0, 2.0]], dtype=jnp.float64)
x_mid = jnp.array([[1.0, 0.0], [2.0, 1.0], [-1.0, 3.0]], dtype=jnp.float64)
a = di.interval(a_mid, a_mid)
x = di.interval(x_mid, x_mid)

a_c_mid = a_mid + 1j * jnp.array([[0.0, 1.0, 0.0], [-1.0, 0.0, 0.5], [0.0, -0.5, 0.0]], dtype=jnp.float64)
x_c_mid = x_mid + 1j * jnp.array([[0.0, 1.0], [-1.0, 0.0], [0.5, -1.0]], dtype=jnp.float64)
a_c = acb_core.acb_box(di.interval(jnp.real(a_c_mid), jnp.real(a_c_mid)), di.interval(jnp.imag(a_c_mid), jnp.imag(a_c_mid)))
x_c = acb_core.acb_box(di.interval(jnp.real(x_c_mid), jnp.real(x_c_mid)), di.interval(jnp.imag(x_c_mid), jnp.imag(x_c_mid)))

In [ ]:
rhs_basic = api.eval_interval('arb_mat_matmul', a, x, mode='basic')
rhs_point = api.eval_point('arb_mat_matmul', a, x)
rhs_rig = api.eval_interval('arb_mat_matmul', a, x, mode='rigorous')

solve_basic = api.eval_interval('arb_mat_solve', a, rhs_basic, mode='basic')
solve_adapt = api.eval_interval('arb_mat_solve', a, rhs_basic, mode='adaptive')
solve_rig = api.eval_interval('arb_mat_solve', a, rhs_basic, mode='rigorous')

print('rhs_point:')
print(rhs_point)
print('solve_basic midpoint:')
print(di.midpoint(solve_basic))
print('solve_adapt shape:', solve_adapt.shape)
print('solve_rig shape:', solve_rig.shape)

In [ ]:
lu_plan = arb_mat.arb_mat_dense_lu_solve_plan_prepare(a)
lu_solution = arb_mat.arb_mat_dense_lu_solve_plan_apply(lu_plan, rhs_basic)
cache = arb_mat.arb_mat_dense_matvec_plan_prepare(a)
vec = arb_mat.arb_mat_diag(x)
cached = arb_mat.arb_mat_dense_matvec_plan_apply(cache, vec)
transpose = arb_mat.arb_mat_transpose(a)
diag = arb_mat.arb_mat_diag(a)
diag_matrix = arb_mat.arb_mat_diag_matrix(diag)

print('lu_solution midpoint:')
print(di.midpoint(lu_solution))
print('cached midpoint:')
print(di.midpoint(cached))
print('transpose midpoint:')
print(di.midpoint(transpose))
print('diag midpoint:', di.midpoint(diag))
print('diag_matrix midpoint:')
print(di.midpoint(diag_matrix))

In [ ]:
rhs_c = api.eval_interval('acb_mat_matmul', a_c, x_c, mode='basic')
solve_c = api.eval_interval('acb_mat_solve', a_c, rhs_c, mode='basic')
lu_plan_c = acb_mat.acb_mat_dense_lu_solve_plan_prepare(a_c)
lu_solution_c = acb_mat.acb_mat_dense_lu_solve_plan_apply(lu_plan_c, rhs_c)
transpose_c = acb_mat.acb_mat_transpose(a_c)
ctranspose_c = acb_mat.acb_mat_conjugate_transpose(a_c)
diag_c = acb_mat.acb_mat_diag(a_c)

print('complex solve midpoint:')
print(acb_core.acb_midpoint(solve_c))
print('complex lu reuse midpoint:')
print(acb_core.acb_midpoint(lu_solution_c))
print('complex transpose midpoint:')
print(acb_core.acb_midpoint(transpose_c))
print('complex conjugate transpose midpoint:')
print(acb_core.acb_midpoint(ctranspose_c))
print('complex diag midpoint:')
print(acb_core.acb_midpoint(diag_c))